# 03 · How Spark executes the analysis

**Central question:** how does Spark turn the operations we write into a distributed execution?

This notebook follows real queries from `02_analysis.ipynb` through every step: the logical plan,
the Catalyst optimizer, the physical plan, and finally the jobs, stages and tasks that run on the cores.

| Section | Query | Project requirement |
|---|---|---|
| 1 | Q2, top districts by incidence, in the DataFrame API **and** in Spark SQL | 8 (both APIs, same result) · 9 (plan 1) |
| 2 | Q5, cases in districts under a rain emergency (`left_semi` join) | 9 (plan 2) |
| 3 | Q1, cases per year: from transformations to jobs, stages and tasks | 11 (lazy evaluation, DAG, stages, tasks) |
| 4 | The shuffles of these queries, measured | 10 (shuffle) |

Spark runs in local mode: one JVM where the *driver* plans the work and the *executor* threads (one per
core) run the tasks. The same plans would run unchanged on a cluster.

In [1]:
import contextlib
import io
import json
import re
import time
import urllib.request
from pathlib import Path

from pyspark.sql import SparkSession, Window, functions as F

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "catalog.yaml").exists())
PROCESSED = ROOT / "data" / "processed"

spark = (SparkSession.builder
         .master("local[*]")
         .appName("dengue-execution-plans")
         .config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.sql.maxMetadataStringLength", 1000)  # don't truncate file paths and filters in plans
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

cases, population, emergencies = (spark.read.parquet(str(PROCESSED / name))
                                  for name in ["dengue_cases", "population", "water_emergencies"])

for key in ["spark.sql.shuffle.partitions", "spark.sql.adaptive.enabled", "spark.sql.autoBroadcastJoinThreshold"]:
    print(f"{key:40} {spark.conf.get(key)}")
print(f"{'cores (default parallelism)':40} {sc.defaultParallelism}")
print(f"{'Spark UI':40} {sc.uiWebUrl}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/01 20:07:38 WARN Utils: Your hostname, Raymonds-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.18.58 instead (on interface en0)
26/11/01 20:07:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/01 20:07:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


spark.sql.shuffle.partitions             200
spark.sql.adaptive.enabled               true
spark.sql.autoBroadcastJoinThreshold     10485760b
cores (default parallelism)              12
Spark UI                                 http://192.168.18.58:4040


In [2]:
def plan_text(df, mode="formatted"):
    # The text that df.explain(mode) prints, captured as a string.
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode=mode)
    return buffer.getvalue()


def plan_section(df, name):
    # One section of explain("extended"): "Parsed Logical Plan", "Optimized Logical Plan", "Physical Plan"...
    return plan_text(df, "extended").split(f"== {name} ==\n")[1].split("\n== ")[0]


def without_ids(text):
    # Expression ids (cases#23L) and plan ids differ between two separately built plans.
    return re.sub(r"#\d+L?|plan_id=\d+", "", text)


def stage_metrics(job_group):
    # Stages run for a job group, read from the Spark UI REST API (the same numbers the UI shows).
    time.sleep(1)  # let the UI listener record the last events
    base = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}"
    jobs = [j for j in json.load(urllib.request.urlopen(f"{base}/jobs")) if j.get("jobGroup") == job_group]
    print(f"{'job':>3} {'stage':>5} {'status':9} {'tasks':>5} {'input rows':>11} {'shuffle write rows':>18} "
          f"{'shuffle read rows':>17} {'shuffle write bytes':>19}")
    for job in sorted(jobs, key=lambda j: j["jobId"]):
        for stage_id in sorted(job["stageIds"]):
            s = json.load(urllib.request.urlopen(f"{base}/stages/{stage_id}"))[0]
            print(f"{job['jobId']:>3} {stage_id:>5} {s['status']:9} {s['numTasks']:>5} {s['inputRecords']:>11,} "
                  f"{s['shuffleWriteRecords']:>18,} {s['shuffleReadRecords']:>17,} {s['shuffleWriteBytes']:>19,}")

## 1 · Plan 1: the same query in the DataFrame API and in Spark SQL

Q2 of the analysis: for each year 2022–2025, the five districts (10,000+ people) with the highest
incidence. Cases are counted per district and year, joined to population, ranked with a window and
sorted. The two versions below express exactly the same logic, one with DataFrame methods and one as
SQL text over temporary views.

In [3]:
MIN_POPULATION = 10_000

district_cases = (cases.filter(F.col("year").between(2022, 2025))
                  .groupBy("year", "ubigeo").agg(F.count("*").alias("cases")))
district_population = (population.groupBy("year", "ubigeo", "department", "province", "district")
                       .agg(F.sum("population").alias("population")))
by_year = Window.partitionBy("year").orderBy(F.desc(F.col("cases") / F.col("population")))

top_districts_df = (district_cases
                    .join(district_population, ["year", "ubigeo"], "left")
                    .filter(F.col("population") >= MIN_POPULATION)
                    .select("year", "department", "province", "district", "cases", "population",
                            F.round(F.col("cases") / F.col("population") * 100000, 1).alias("incidence_100k"),
                            F.dense_rank().over(by_year).alias("rank"))
                    .filter(F.col("rank") <= 5)
                    .select("year", "rank", "department", "province", "district", "cases", "population", "incidence_100k")
                    .orderBy("year", "rank"))

In [4]:
# Spark SQL needs the tables registered as views
cases.createOrReplaceTempView("dengue_cases")
population.createOrReplaceTempView("population")

top_districts_sql = spark.sql(f"""
    WITH district_cases AS (
        SELECT year, ubigeo, COUNT(*) AS cases
        FROM dengue_cases
        WHERE year BETWEEN 2022 AND 2025
        GROUP BY year, ubigeo
    ),
    district_population AS (
        SELECT year, ubigeo, department, province, district, SUM(population) AS population
        FROM population
        GROUP BY year, ubigeo, department, province, district
    ),
    ranked AS (
        SELECT year, department, province, district, cases, population,
               ROUND(cases / population * 100000, 1) AS incidence_100k,
               DENSE_RANK() OVER (PARTITION BY year ORDER BY cases / population DESC) AS rank
        FROM district_cases
        LEFT JOIN district_population USING (year, ubigeo)
        WHERE population >= {MIN_POPULATION}
    )
    SELECT year, rank, department, province, district, cases, population, incidence_100k
    FROM ranked
    WHERE rank <= 5
    ORDER BY year, rank
""")

### Same result

`exceptAll` in both directions returns the rows present in one result and missing from the other
(duplicates included). Both must be empty.

In [5]:
only_in_df = top_districts_df.exceptAll(top_districts_sql).count()
only_in_sql = top_districts_sql.exceptAll(top_districts_df).count()
print(f"rows: DataFrame API {top_districts_df.count()}, Spark SQL {top_districts_sql.count()}")
print(f"only in DataFrame API: {only_in_df} | only in Spark SQL: {only_in_sql}")
assert only_in_df == 0 and only_in_sql == 0
top_districts_sql.show(20, truncate=False)

rows: DataFrame API 20, Spark SQL 20
only in DataFrame API: 0 | only in Spark SQL: 0


+----+----+-----------+-----------------+--------------------+-----+----------+--------------+
|year|rank|department |province         |district            |cases|population|incidence_100k|
+----+----+-----------+-----------------+--------------------+-----+----------+--------------+
|2022|1   |ÁNCASH     |CASMA            |CASMA               |1927 |38179     |5047.3        |
|2022|2   |PIURA      |MORROPÓN         |MORROPÓN            |683  |14420     |4736.5        |
|2022|3   |AMAZONAS   |BAGUA            |BAGUA               |1342 |29254     |4587.4        |
|2022|4   |CUSCO      |LA CONVENCIÓN    |SANTA ANA           |1775 |41669     |4259.8        |
|2022|5   |JUNÍN      |CHANCHAMAYO      |SAN RAMÓN           |1110 |29710     |3736.1        |
|2023|1   |PIURA      |PAITA            |LA HUACA            |1890 |13958     |13540.6       |
|2023|2   |ÁNCASH     |SANTA            |NEPEÑA              |1428 |15854     |9007.2        |
|2023|3   |PIURA      |SULLANA          |MIGUEL CH

### Same plan

Each plan is compared after removing the internal expression ids (`cases#23L`), which are numbered
separately every time a plan is built.

In [6]:
for name in ["Parsed Logical Plan", "Analyzed Logical Plan", "Optimized Logical Plan", "Physical Plan"]:
    same = without_ids(plan_section(top_districts_df, name)) == without_ids(plan_section(top_districts_sql, name))
    print(f"{name:24} identical: {same}")
same_formatted = without_ids(plan_text(top_districts_df)) == without_ids(plan_text(top_districts_sql))
label = 'explain("formatted")'
print(f"{label:24} identical: {same_formatted}")
assert same_formatted

Parsed Logical Plan      identical: False
Analyzed Logical Plan    identical: False
Optimized Logical Plan   identical: True
Physical Plan            identical: True
explain("formatted")     identical: True


**Same result and same plan.** The parsed and analyzed plans differ, as expected: Spark SQL starts from text that the parser turns into a tree with CTEs and a `USING` join, while the DataFrame API builds the tree directly from method calls. From the **optimized logical plan** onwards the two are identical, and so is the physical plan Spark will run. Catalyst and the rest of the engine do not care which API wrote the query.

### Logical plan after Catalyst

The optimized logical plan (from `explain("extended")`), before Spark decides *how* to run it:

In [7]:
print(plan_section(top_districts_sql, "Optimized Logical Plan"))

Sort [year#1351 ASC NULLS FIRST, rank#1367 ASC NULLS FIRST], true
+- Project [year#1351, rank#1367, department#1355, province#1356, district#1357, cases#1352L, population#1361L, incidence_100k#1362]
   +- Filter (rank#1367 <= 5)
      +- Window [dense_rank(_w0#1363) windowspecdefinition(year#1351, _w0#1363 DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS rank#1367], [year#1351], [_w0#1363 DESC NULLS LAST]
         +- WindowGroupLimit [year#1351], [_w0#1363 DESC NULLS LAST], dense_rank(_w0#1363), 5
            +- Project [year#1351, department#1355, province#1356, district#1357, cases#1352L, population#1361L, round(((cast(cases#1352L as double) / cast(population#1361L as double)) * 100000.0), 1) AS incidence_100k#1362, (cast(cases#1352L as double) / cast(population#1361L as double)) AS _w0#1363]
               +- Join Inner, ((year#1351 = year#1353) AND (ubigeo#1339 = ubigeo#1354))
                  :- Aggregate [year#1351, ubigeo#1339], [year#13

### Physical plan

In [8]:
top_districts_df.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (25)
+- Sort (24)
   +- Exchange (23)
      +- Project (22)
         +- Filter (21)
            +- Window (20)
               +- WindowGroupLimit (19)
                  +- Sort (18)
                     +- Exchange (17)
                        +- WindowGroupLimit (16)
                           +- Sort (15)
                              +- Project (14)
                                 +- BroadcastHashJoin Inner BuildRight (13)
                                    :- HashAggregate (5)
                                    :  +- Exchange (4)
                                    :     +- HashAggregate (3)
                                    :        +- Filter (2)
                                    :           +- Scan parquet  (1)
                                    +- BroadcastExchange (12)
                                       +- Filter (11)
                                          +- HashAggregate (10)
                                             +- 

### Reading plan 1

**Logical plan: what the query asks.** Count cases per (year, district), sum population per (year, district), LEFT JOIN the two, keep districts with 10,000+ people, rank districts within each year, keep the top 5 and sort.

**What Catalyst changed** (each change is visible in the plans above):

1. **Outer join → inner join.** The optimized plan says `Join Inner`. The filter `population >= 10000` drops every row with a null population, which are exactly the rows a LEFT JOIN adds, so the result equals an inner join and Catalyst rewrites it.
2. **Inferred filters.** The year range was only written for cases, but the join says `cases.year = population.year`, so Catalyst also applies it to population. The population scan shows `PushedFilters: [GreaterThanOrEqual(year,2022), LessThanOrEqual(year,2025), ...]`, plus `isnotnull` filters on the join keys.
3. **Filter below the join.** `population >= 10000` runs right after the population aggregation, before the join (operator 11).
4. **Partition pruning.** `dengue_cases` is stored in one folder per year, so `PartitionFilters: [... year >= 2022, year <= 2025 ...]` means Spark only opens 4 of the 26 year folders.
5. **Dynamic partition pruning.** `dynamicpruningexpression(year IN dynamicpruning...)` adds, at runtime, the years that survive on the population side as one more filter on the case folders (the subquery at the end of the plan).
6. **Column pruning.** The case scan reads `struct<ubigeo:string>`, a single column out of 14 (the year comes from the folder name).
7. **Top-N before the shuffle.** `WindowGroupLimit` keeps only the 5 best districts per year inside each partition *before* the window's shuffle (Partial, operator 16) and again after it (Final, operator 19), so the shuffle moves a few rows per year instead of every district.

**Physical plan: how it runs.**

- **Two-phase aggregation.** `HashAggregate (partial_count)` → `Exchange hashpartitioning(year, ubigeo, 200)` → `HashAggregate (count)` for cases, and the same for population (operators 6–10).
- **Broadcast join.** After filtering, the aggregated population is a few thousand rows, far below the 10 MB `autoBroadcastJoinThreshold`. A `BroadcastExchange` copies it to every task and the join runs as `BroadcastHashJoin Inner BuildRight`, so the case side is not shuffled for the join.
- **Window.** `Exchange hashpartitioning(year)` gathers each year in one partition, then `Sort` → `Window` (`dense_rank`) → `Filter (rank <= 5)`.
- **Global sort.** `Exchange rangepartitioning(year, rank)` → `Sort` for the final `orderBy`.
- **Adaptive execution.** `AdaptiveSparkPlan isFinalPlan=false`: AQE may still change parts of the plan once real data sizes are known.

In total this plan has **four shuffles** (the four `Exchange` operators with hash or range partitioning) and **one broadcast**.

## 2 · Plan 2: a semi join that needs no shuffle

Q5 counted the 2023–2024 cases that happened in districts under a rain emergency. A district appears
once per decree and per extension, so an inner join against the decree rows repeats each case several
times, while a `left_semi` join only checks that a match exists and keeps each case once.

In [9]:
YEARS = [2023, 2024]
cases_2324 = cases.filter(F.col("year").isin(YEARS)).select("year", "ubigeo")
rain_decrees = (emergencies.filter((F.col("emergency_type") == "rain") & F.col("year").isin(YEARS))
                .select("year", "ubigeo"))

semi = cases_2324.join(rain_decrees, ["year", "ubigeo"], "left_semi")
inner = cases_2324.join(rain_decrees, ["year", "ubigeo"], "inner")
print(f"left_semi: {semi.count():>9,} rows | inner: {inner.count():>9,} rows | cases 2023-2024: {cases_2324.count():>9,}")
semi.explain("formatted")

left_semi:   352,213 rows | inner: 1,539,044 rows | cases 2023-2024:   528,172
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- BroadcastHashJoin LeftSemi BuildRight (7)
      :- Filter (2)
      :  +- Scan parquet  (1)
      +- BroadcastExchange (6)
         +- Project (5)
            +- Filter (4)
               +- Scan parquet  (3)


(1) Scan parquet 
Output [2]: [ubigeo#1, year#13]
Batched: true
Location: InMemoryFileIndex [file:/Users/raymondnegron/Documents/MAE - Data Science/MAE-CDE-apache-spark/data/processed/dengue_cases]
PartitionFilters: [year#13 IN (2023,2024), isnotnull(year#13), dynamicpruningexpression(year#13 IN dynamicpruning#1394)]
PushedFilters: [IsNotNull(ubigeo)]
ReadSchema: struct<ubigeo:string>

(2) Filter
Input [2]: [ubigeo#1, year#13]
Condition : isnotnull(ubigeo#1)

(3) Scan parquet 
Output [3]: [ubigeo#22, year#26, emergency_type#35]
Batched: true
Location: InMemoryFileIndex [file:/Users/raymondnegron/Documents/MAE - Data Science/MAE-CDE-apache-

In [10]:
# The inner join: same strategy, different join type
inner.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [year#13, ubigeo#1]
   +- BroadcastHashJoin [year#13, ubigeo#1], [year#26, ubigeo#22], Inner, BuildRight, false, false
      :- Filter isnotnull(ubigeo#1)
      :  +- FileScan parquet [ubigeo#1,year#13] Batched: true, DataFilters: [isnotnull(ubigeo#1)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/raymondnegron/Documents/MAE - Data Science/MAE-CDE-apache-spark/data/processed/dengue_cases], PartitionFilters: [year#13 IN (2023,2024), isnotnull(year#13), dynamicpruningexpression(year#13 IN dynamicpruning#1395)], PushedFilters: [IsNotNull(ubigeo)], ReadSchema: struct<ubigeo:string>
      :        +- SubqueryAdaptiveBroadcast dynamicpruning#1395, [0], true, Project [year#26, ubigeo#22], [year#26, ubigeo#22]
      :           +- AdaptiveSparkPlan isFinalPlan=false
      :              +- Project [year#26, ubigeo#22]
      :                 +- Filter ((((isnotnull(emergency_type#35) AND (emergency_type#35 

### Reading plan 2

The two joins differ only in their join type, yet the inner join returns **1,539,044 rows** (almost three times the 528,172 cases), while the semi join returns the correct **352,213**.

**Logical plan.** Keep 2023–2024 cases; keep rain decrees of 2023–2024; join on (year, district).

**What Catalyst changed.**

- **Partition pruning.** `PartitionFilters: [year IN (2023,2024)]` reads 2 of the 26 case folders.
- **Filter pushdown.** The decree filters reach the Parquet reader: `PushedFilters: [EqualTo(emergency_type,rain), In(year, [2023,2024]), ...]`, so row groups that cannot match are skipped while reading.
- **Column pruning.** Cases read only `ubigeo`; decrees read 3 of their 15 columns.
- **Dynamic partition pruning.** The years that remain in the decree side become an extra runtime filter on the case folders.

**Physical plan: no shuffle at all.** The filtered decree rows are small, so Spark chooses `BroadcastHashJoin LeftSemi BuildRight`. The decree side is built into a hash table once and broadcast. Each task scans its slice of cases and only checks whether the (year, district) key exists in that table, emitting each case at most once. There is **no `Exchange`** in the plan: every task joins its own data without exchanging rows with other tasks. The inner join uses the same strategy, but emits one row per matching decree row, which is why its count inflates.

## 3 · From transformations to jobs, stages and tasks

The Q1 aggregation (cases per year) is defined step by step. A job group tags everything Spark runs
for it, so it can be traced afterwards.

In [11]:
sc.setJobGroup("q1", "cases per year")

start = time.perf_counter()
cases_per_year = (cases.groupBy("year")
                  .agg(F.count("*").alias("cases"))
                  .orderBy("year"))
print(f"transformations defined in {time.perf_counter() - start:.3f} s")
print("jobs run so far:", sc.statusTracker().getJobIdsForGroup("q1"))

transformations defined in 0.008 s
jobs run so far: []


**Lazy evaluation.** Defining the query took a few milliseconds and started **no job**: `groupBy`, `agg` and `orderBy` are *transformations*, and Spark only adds them to the logical plan. Nothing is read until an *action* (`collect`, `count`, `show`, `write`) asks for a result.

In [12]:
start = time.perf_counter()
rows = cases_per_year.collect()  # the action
print(f"collect() returned {len(rows)} rows in {time.perf_counter() - start:.2f} s")
print("jobs run:", sorted(sc.statusTracker().getJobIdsForGroup("q1")))
stage_metrics("q1")

collect() returned 26 rows in 0.14 s
jobs run: [48, 49, 50, 51]


job stage status    tasks  input rows shuffle write rows shuffle read rows shuffle write bytes
 48   106 COMPLETE      9   1,065,274                 26                 0               1,664
 49   107 SKIPPED       9           0                  0                 0                   0
 49   108 COMPLETE      1           0                  0                26                   0
 50   109 SKIPPED       9           0                  0                 0                   0
 50   110 COMPLETE      1           0                 26                26               1,664
 51   111 SKIPPED       9           0                  0                 0                   0
 51   112 SKIPPED       1           0                  0                 0                   0
 51   113 COMPLETE      1           0                  0                26                   0


In [13]:
# After execution the plan shows what adaptive query execution (AQE) decided at runtime
cases_per_year.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (18)
+- == Final Plan ==
   ResultQueryStage (12)
   +- * Sort (11)
      +- AQEShuffleRead (10), coalesced
         +- ShuffleQueryStage (9), Statistics(sizeInBytes=624.0 B, rowCount=26)
            +- Exchange (8)
               +- * HashAggregate (7)
                  +- AQEShuffleRead (6), coalesced
                     +- ShuffleQueryStage (5), Statistics(sizeInBytes=624.0 B, rowCount=26)
                        +- Exchange (4)
                           +- * HashAggregate (3)
                              +- * ColumnarToRow (2)
                                 +- Scan parquet  (1)
+- == Initial Plan ==
   Sort (17)
   +- Exchange (16)
      +- HashAggregate (15)
         +- Exchange (14)
            +- HashAggregate (13)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [year#13]
Batched: true
Location: InMemoryFileIndex [file:/Users/raymondnegron/Documents/MAE - Data Science/MAE-CDE-apache-spark/data/processed/dengue_cases]

**From plan to DAG, stages and tasks.** The action turned the physical plan into a **DAG of stages**: the scheduler cuts the plan at every `Exchange`, because a stage can only start once the data it needs has been redistributed. For this query (two exchanges: the `groupBy` and the `orderBy`):

| Stage (in the table above) | Tasks | What it does |
|---|---|---|
| first stage | 9 | Reads the 1,065,274 cases (only the `year` column; `ReadSchema: struct<>`, since year is a folder name), counts per year inside each task (`partial_count`) and writes **26 partial rows** to the shuffle |
| next | 1 | Reads the shuffle and **samples the sort keys**: `orderBy` needs range boundaries before it can redistribute |
| next | 1 | Finishes the count per year (`HashAggregate`) and writes the 26 totals, partitioned by range for the sort |
| last | 1 | Reads the ranges, sorts them and returns the 26 rows to the driver |

- **Tasks.** One per partition: 9 for the scan, because Spark packed the Parquet files into 9 input splits for the 12 cores, and 1 for the later stages.
- **Several jobs for one action.** With AQE, each shuffle stage is submitted as its own job, so one `collect()` produced 4 jobs. Later jobs show the stages already computed as **SKIPPED**: they reuse the shuffle files instead of recomputing them.
- **Final plan.** It shows what AQE decided at runtime. `ShuffleQueryStage` reports real statistics (624 bytes, 26 rows), and `AQEShuffleRead coalesced` merges the 200 default shuffle partitions into 1. Operators marked `*` share a `codegen id`: **whole-stage code generation** compiles each chain into a single Java function, so rows are not passed one operator at a time.

To see the DAG graphically, run the notebook interactively and open the Spark UI printed in the setup cell (**Jobs → DAG Visualization**; the **SQL / DataFrame** tab draws the physical plan with row counts per operator).

## 4 · Shuffle: why the data has to move, and how much moves

The same Q1 aggregation is run again with adaptive query execution turned off, to see the raw
behaviour of the shuffle with Spark's default 200 partitions.

In [14]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
sc.setJobGroup("q1-no-aqe", "cases per year without AQE")

start = time.perf_counter()
cases.groupBy("year").agg(F.count("*").alias("cases")).orderBy("year").collect()
print(f"without AQE: {time.perf_counter() - start:.2f} s")
stage_metrics("q1-no-aqe")

spark.conf.set("spark.sql.adaptive.enabled", "true")
sc.setJobGroup("other", "rest of the notebook")

without AQE: 0.24 s


job stage status    tasks  input rows shuffle write rows shuffle read rows shuffle write bytes
 52   114 COMPLETE      9   1,065,274                 26                 0               1,664
 52   115 COMPLETE    200           0                  0                26                   0
 53   116 SKIPPED       9           0                  0                 0                   0
 53   117 COMPLETE    200           0                 26                26               1,664
 53   118 COMPLETE     26           0                  0                26                   0


**Why the shuffle is needed.** The cases of a given year are spread over files read by different tasks. To produce one total per year, every partial count for that year must end up in the **same partition**. The `Exchange hashpartitioning(year)` sends each row to partition `hash(year) mod N`: tasks write their output to local disk, grouped by destination partition (*shuffle write*), and the next stage's tasks fetch their part from every writer (*shuffle read*). `orderBy` needs the same kind of movement (`rangepartitioning`), so that partition *i* only holds keys smaller than partition *i + 1*. A shuffle is the most expensive step of a Spark job: it means disk I/O, serialization and, on a cluster, network traffic, and it is also the boundary between stages.

**What the measurements show.**

- **Partial aggregation shrinks the shuffle.** The map side counts per year *before* shuffling, so 1,065,274 input rows become **26 shuffled rows (1,664 bytes)**. Without that combine step, the whole column would cross the network.
- **The default 200 partitions are far too many here.** Without AQE, **200 tasks** process the 26 shuffled rows, twice (once to sample the sort ranges and once to finish the count), and the sort adds 26 more: mostly empty tasks that only add scheduling overhead. With AQE, Spark measured the 624-byte shuffle and **coalesced it into 1 partition (1 task)**.
- **Plan 1 needs four shuffles** (two aggregations, the window, the sort), but **not for its join**: broadcasting the small population table avoided shuffling the cases. **Plan 2 needs none.** Avoiding shuffles is the main lever for making a Spark job faster.

## Summary: how Spark turns our operations into a distributed execution

Following these queries end to end answers the central question:

1. **We describe the result, not the steps.** DataFrame calls or SQL text only build a *logical plan*. Nothing runs yet (lazy evaluation: milliseconds, zero jobs).
2. **Analysis.** Spark resolves table and column names and types against the catalog. A wrong column name fails here, before any data is read (it happened in notebook 01). The DataFrame API and Spark SQL only differ up to this point.
3. **Optimization (Catalyst).** Rule-based rewrites produce the *optimized logical plan*: filters pushed into the Parquet reader, partition and column pruning, filters inferred through joins, outer joins turned into inner joins, top-N limits before window shuffles. Q2 in both APIs reaches the same optimized plan.
4. **Physical planning.** Spark chooses *how* to run each operator: two-phase `HashAggregate`, `BroadcastHashJoin` when one side is small instead of shuffling both sides, `Exchange` wherever data must be regrouped. Whole-stage code generation compiles chains of operators into Java code.
5. **Scheduling.** An action turns the physical plan into a DAG of **stages**, cut at every `Exchange`. AQE re-plans between stages with real sizes (coalescing 200 shuffle partitions into 1).
6. **Execution.** Each stage runs as **tasks**, one per partition, in parallel on the executor cores (9 tasks on 12 cores for the scan). Between stages, the **shuffle** writes and reads the redistributed data.

In short, Spark turns a declarative query into an optimized plan, splits it into stages at the points where data has to move, and runs each stage as parallel tasks. The same code would run on a cluster with thousands of cores without changes.

In [15]:
spark.stop()